# 05 · Convert AERONET files to parquet (any product)

One notebook for every AERONET Version 3 product delivered as text: sun AOD, lunar AOD, almucantar and hybrid inversions; all points, daily or monthly.

**It changes the format, never the data.** For the product picked below it

1. reads the source: **one file with every site** (`All_Sites_Times_Daily_Averages_AOD20.dat`) or **a folder with one file per site** (`*.lev15`, `*.lev20`, `*.all`);
2. turns AERONET's missing marker `-999` into empty (NaN);
3. drops the columns that are empty at every site (wavelengths nobody measures, the `..._Empty` placeholders) and their count columns;
4. turns the date and time text into real dates (`datetime_utc`, `date`, `year`, `month`);
5. cleans the column names so they work in pandas / SQL (`Precipitable_Water(cm)` → `Precipitable_Water_cm`);
6. writes one parquet into `DATA/PROCESSED/`.

No value is changed, no row is removed, rows stay in AERONET's order. The PI contact lines of the file headers are not kept.

The files are read **one site at a time, twice** (first to learn the columns, then to write), so even the 11 GB lunar product needs little memory.

In [1]:
from pathlib import Path
import re

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

/home/ubuntu/miniconda3/envs/gridding_code/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Pick the product

Uncomment **one** line, run the notebook from the top, then comment it again and pick the next.
`SOURCE` is a file or a folder under `DATA/RAW/`; `OUT` is the parquet written into `DATA/PROCESSED/`.

In [2]:
# ── sun AOD
# SOURCE, OUT = "data/All_Sites_Times_Daily_Averages_AOD20.dat", "AERONET_AOD_L2_Daily_V3.parquet"
# SOURCE, OUT = "data/All_Sites_Times_Daily_Averages_AOD15.dat", "AERONET_AOD_L15_Daily_V3.parquet"
# SOURCE, OUT = "data/AOD_Level20_Monthly_V3",                    "AERONET_AOD_L2_Monthly_V3.parquet"
# SOURCE, OUT = "data/AOD_Level15_Monthly_V3",                    "AERONET_AOD_L15_Monthly_V3.parquet"

# ── lunar AOD
# SOURCE, OUT = "data/AOD_Lunar_Level20_All_Points_V3",           "AERONET_Lunar_AOD_L2_AllPoints_V3.parquet"
# SOURCE, OUT = "data/AOD_Lunar_Level15_All_Points_V3",           "AERONET_Lunar_AOD_L15_AllPoints_V3.parquet"

# ── hybrid inversions
# SOURCE, OUT = "INV_HYB_Level2_Daily_V3",                        "AERONET_INV_Hybrid_L2_Daily_V3.parquet"
# SOURCE, OUT = "data/INV_HYB_Level2_Monthly_V3",                 "AERONET_INV_Hybrid_L2_Monthly_V3.parquet"
# SOURCE, OUT = "data/INV_HYB_Level15_Monthly_V3",                "AERONET_INV_Hybrid_L15_Monthly_V3.parquet"

# ── almucantar inversions
# SOURCE, OUT = "data/INV_Level2_All_Points_V3",                  "AERONET_INV_Almucantar_L2_AllPoints_V3.parquet"
# SOURCE, OUT = "data/INV_Level15_All_Points_V3",                 "AERONET_INV_Almucantar_L15_AllPoints_V3.parquet"
# SOURCE, OUT = "data/INV_Level15_Daily_V3",                      "AERONET_INV_Almucantar_L15_Daily_V3.parquet"
SOURCE, OUT = "data/INV_Level15_Monthly_V3",                    "AERONET_INV_Almucantar_L15_Monthly_V3.parquet"

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "process_data":          # this notebook lives in notebooks/process_data/
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

SOURCE      = PROJECT_ROOT / "DATA" / "RAW" / SOURCE
OUT_PARQUET = PROJECT_ROOT / "DATA" / "PROCESSED" / OUT

print(f"SOURCE      : {SOURCE}  ({'one file with every site' if SOURCE.is_file() else 'a folder with one file per site'})")
print(f"OUT_PARQUET : {OUT_PARQUET}")

SOURCE      : /home/ubuntu/ai_for_aeronet/DATA/RAW/data/INV_Level15_Monthly_V3  (a folder with one file per site)
OUT_PARQUET : /home/ubuntu/ai_for_aeronet/DATA/PROCESSED/AERONET_INV_Almucantar_L15_Monthly_V3.parquet


## 2 · Read the source, one site at a time

Every AERONET text file has 6 lines of notes (line 2 is the site name), then a normal CSV with the column names on line 7.

In [4]:
MISSING = -999.0

def read(path):
    return pd.read_csv(path, skiprows=6, encoding="latin-1", low_memory=False)


def sites():
    # gives (site name, the rows of that site with AERONET's own column names), one site after the other
    if SOURCE.is_file():                                                   # one file with every site
        for site, df in read(SOURCE).groupby("AERONET_Site", sort=False):
            yield site, df
    else:                                                                  # a folder with one file per site
        for path in sorted(p for p in SOURCE.rglob("*") if p.suffix in (".lev15", ".lev20", ".all")):
            with open(path, encoding="latin-1") as fh:
                next(fh)
                site = next(fh).strip()                                    # line 2
            yield site, read(path)

## 3 · First pass: learn the columns

Sites do not all have the same columns (the inversion files come with different filter sets), and a column can be whole numbers at one site and text or decimals at another.
So the first pass only looks: which columns exist, how many values each one holds, and what type each one needs. Nothing is written yet.

In [5]:
SITE_COLUMNS = ["AERONET_Site", "Site", "AERONET_Site_Name"]      # the site name repeated in every row, under three names

columns  = []              # every column seen, in AERONET's order
filled   = pd.Series(dtype="float64")     # per column: how many values are not missing
text     = set()           # columns that hold text at some site
decimal  = set()           # columns that hold decimals (or a missing value) at some site
n_rows = n_sites = mismatch = 0

for site, df in tqdm(sites(), desc="first pass"):
    df = df.mask(df == MISSING)                                    # -999 -> NaN
    columns  = list(dict.fromkeys(columns + list(df.columns)))
    filled   = filled.add(df.notna().sum(), fill_value=0)
    numbers  = set(df.select_dtypes("number").columns)
    text    |= set(df.columns) - numbers
    decimal |= numbers - set(df.select_dtypes("integer").columns)
    n_rows  += len(df)
    n_sites += 1
    for c in SITE_COLUMNS:
        if c in df.columns:
            mismatch += int((df[c] != site).sum())

print(f"{n_sites:,} sites, {n_rows:,} rows, {len(columns)} columns")
print(f"rows whose site name differs from the file's site name: {mismatch}")

first pass: 1513it [00:32, 45.98it/s]

1,513 sites, 284,850 rows, 494 columns
rows whose site name differs from the file's site name: 0


In [6]:
def counted(c):
    # "N[AOD_500nm]" -> "AOD_500nm": the column a count column belongs to (None for every other column)
    m = re.fullmatch(r"(?:N|NUM_DAYS|NUM_POINTS)\[(.+)\]", c)
    return m.group(1) if m else None

drop  = [c for c in columns if filled[c] == 0 or "Empty" in c]            # empty at every site, or a placeholder
drop += [c for c in columns if counted(c) in drop and c not in drop]      # ... and their count columns
keep  = [c for c in columns if c not in drop]
dtype = {c: "string" if c in text else "float64" if c in decimal else "Int32" for c in keep}

print(f"dropping {len(drop)} columns, keeping {len(keep)}:")
print(drop)

dropping 0 columns, keeping 494:
[]


## 4 · Tidy one site

* the site name becomes the first column, `AERONET_Site_Name`, in every product (the monthly files have it only in their header);
* the date and time text become `datetime_utc`, `date`, `year`, `month`; a monthly file has `Month` (`1993-MAY`) instead, which becomes `date` (the first day of that month), `year`, `month`;
* the same thing gets the same name in every product: `Latitude(degrees)` / `Latitude(Degrees)` → `Site_Latitude_Degrees`, `Level` → `Data_Quality_Level`, … (the `SAME` list below);
* the 22 size-distribution bins of the inversion files are headed by their radius in µm (`0.050000`): they become `dVdlnr_0_050000um`;
* every other column keeps AERONET's name, with `( ) [ ] -` → `_` and `(%)` → `_percent`.

In [7]:
SAME = {"Latitude(Degrees)":  "Site_Latitude(Degrees)",  "Latitude(degrees)":  "Site_Latitude(Degrees)",
        "Longitude(Degrees)": "Site_Longitude(Degrees)", "Longitude(degrees)": "Site_Longitude(Degrees)",
        "Elevation(m)":       "Site_Elevation(m)",       "Elevation(meters)":  "Site_Elevation(m)",
        "Level": "Data_Quality_Level", "Inversion_Data_Quality_Level": "Data_Quality_Level",
        "Instrument_Number": "AERONET_Instrument_Number",
        "Day_of_Year(fraction)": "Day_of_Year(Fraction)"}

def clean_col(name):
    name = SAME.get(name, name)
    name = re.sub(r"(^|\[)(\d+\.\d+)(\]|$)", r"\1dVdlnr_\2um\3", name)                    # size bins: 0.050000 -> dVdlnr_0.050000um
    name = re.sub(r"(\d{3})-(\d{3})_Angstrom_Exponent", r"Angstrom_Exponent_\1_\2nm", name)
    name = name.replace("[Polar]", "_Polar").replace("(%)", "_percent")
    name = re.sub(r"[\(\[]", "_", name)
    name = re.sub(r"[\)\]]", "", name)
    name = re.sub(r"[^0-9A-Za-z_]", "_", name)
    return re.sub(r"_+", "_", name).strip("_")

In [8]:
def tidy(site, df):
    df = df.mask(df == MISSING).reindex(columns=keep).astype(dtype)        # same columns and types for every site
    df = df.drop(columns=[c for c in SITE_COLUMNS if c in df.columns])

    front = pd.DataFrame({"AERONET_Site_Name": pd.array([site] * len(df), dtype="string")}, index=df.index)
    if "Month" in df.columns:                                              # monthly file
        when = pd.to_datetime(df.pop("Month"), format="%Y-%b")
    else:
        when = pd.to_datetime(df.pop("Date(dd:mm:yyyy)") + " " + df.pop("Time(hh:mm:ss)"), format="%d:%m:%Y %H:%M:%S")
        front["datetime_utc"] = when
    front["date"]  = when.dt.normalize()
    front["year"]  = when.dt.year.astype("int16")
    front["month"] = when.dt.month.astype("int8")

    df = pd.concat([front, df], axis=1)                                    # site and time first, then AERONET's order
    df.columns = [clean_col(c) for c in df.columns]
    return df

## 5 · Second pass: tidy each site and append it to the parquet

Sites are written in groups of about 100,000 rows (one parquet "row group"): a monthly file of 50 rows would otherwise become its own tiny block,
which makes the parquet big and slow to open. Rows stay in site order, so reading one site later only touches its own block.

In [9]:
ROW_GROUP = 100_000

writer, waiting, waiting_rows = None, [], 0
for i, (site, df) in enumerate(tqdm(sites(), total=n_sites, desc="writing"), 1):
    table = pa.Table.from_pandas(tidy(site, df), preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUT_PARQUET, table.schema, compression="zstd")
    waiting.append(table)
    waiting_rows += table.num_rows
    if waiting_rows >= ROW_GROUP or i == n_sites:                  # enough rows for one block, or the last site
        writer.write_table(pa.concat_tables(waiting))
        waiting, waiting_rows = [], 0
writer.close()

print(f"wrote {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size / 1e6:,.0f} MB)")

writing: 100%|██████████| 1513/1513 [04:26<00:00,  5.67it/s]

wrote /home/ubuntu/ai_for_aeronet/DATA/PROCESSED/AERONET_INV_Almucantar_L15_Monthly_V3.parquet  (74 MB)


## 6 · Sanity checks

Read back from the parquet: only the site and date columns (the big tables do not fit in memory whole), then the first rows.

In [10]:
pf    = pq.ParquetFile(OUT_PARQUET)
check = pq.read_table(OUT_PARQUET, columns=["AERONET_Site_Name", "date"])
dates = check["date"].to_pandas()

print(f"sites      : {len(check['AERONET_Site_Name'].unique()):,}  (first pass: {n_sites:,})")
print(f"rows       : {pf.metadata.num_rows:,}  (first pass: {n_rows:,})")
print(f"columns    : {pf.metadata.num_columns}")
print(f"date range : {dates.min().date()}  →  {dates.max().date()}")

sites      : 1,513  (first pass: 1,513)
rows       : 284,850  (first pass: 284,850)
columns    : 497
date range : 1993-05-01  →  2026-10-01


In [11]:
print("non-missing values per column, % of rows (from the first pass, AERONET's own column names):")
display((filled[keep] / n_rows * 100).round(1).rename("% filled").to_frame().T)

non-missing values per column, % of rows (from the first pass, AERONET's own column names):


,Month,AOD_Coincident_Input[443nm],AOD_Coincident_Input[667nm],AOD_Coincident_Input[865nm],AOD_Coincident_Input[1020nm],Angstrom_Exponent_440-870nm_from_Coincident_Input_AOD,AOD_Extinction-Total[443nm],AOD_Extinction-Total[667nm],AOD_Extinction-Total[865nm],AOD_Extinction-Total[1020nm],...,NUM_POINTS[Spectral_Flux_Up[870nm]],NUM_POINTS[Spectral_Flux_Diffuse[440nm]],NUM_POINTS[Spectral_Flux_Diffuse[675nm]],NUM_POINTS[Spectral_Flux_Diffuse[870nm]],NUM_POINTS[Lidar_Ratio[440nm]],NUM_POINTS[Lidar_Ratio[675nm]],NUM_POINTS[Lidar_Ratio[870nm]],NUM_POINTS[Depolarization_Ratio[440nm]],NUM_POINTS[Depolarization_Ratio[675nm]],NUM_POINTS[Depolarization_Ratio[870nm]]
% filled,100.0,0.6,0.6,0.2,21.4,21.4,0.6,0.6,0.2,21.4,...,99.5,98.2,98.2,99.5,98.2,98.2,99.5,98.2,98.2,99.5


In [12]:
print("the first rows:")
display(pf.read_row_group(0).to_pandas().head())

the first rows:


,AERONET_Site_Name,date,year,month,AOD_Coincident_Input_443nm,AOD_Coincident_Input_667nm,AOD_Coincident_Input_865nm,AOD_Coincident_Input_1020nm,Angstrom_Exponent_440_870nm_from_Coincident_Input_AOD,AOD_Extinction_Total_443nm,...,NUM_POINTS_Spectral_Flux_Up_870nm,NUM_POINTS_Spectral_Flux_Diffuse_440nm,NUM_POINTS_Spectral_Flux_Diffuse_675nm,NUM_POINTS_Spectral_Flux_Diffuse_870nm,NUM_POINTS_Lidar_Ratio_440nm,NUM_POINTS_Lidar_Ratio_675nm,NUM_POINTS_Lidar_Ratio_870nm,NUM_POINTS_Depolarization_Ratio_440nm,NUM_POINTS_Depolarization_Ratio_675nm,NUM_POINTS_Depolarization_Ratio_870nm
0,AAOT,2018-02-01,2018,2,0.173896,0.104537,0.069117,0.052912,1.413724,0.174818,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,AAOT,2018-03-01,2018,3,0.260874,0.147331,0.093045,0.068882,1.506527,0.261693,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,AAOT,2018-04-01,2018,4,0.226634,0.152511,0.124099,0.109342,1.030219,0.227814,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,AAOT,2018-05-01,2018,5,0.258638,0.148970,0.108980,0.091654,1.392814,0.259343,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,AAOT,2018-06-01,2018,6,0.187612,0.104478,0.073655,0.060866,1.509555,0.188541,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [13]:
print(pf.schema_arrow.names)

['AERONET_Site_Name', 'date', 'year', 'month', 'AOD_Coincident_Input_443nm', 'AOD_Coincident_Input_667nm', 'AOD_Coincident_Input_865nm', 'AOD_Coincident_Input_1020nm', 'Angstrom_Exponent_440_870nm_from_Coincident_Input_AOD', 'AOD_Extinction_Total_443nm', 'AOD_Extinction_Total_667nm', 'AOD_Extinction_Total_865nm', 'AOD_Extinction_Total_1020nm', 'AOD_Extinction_Fine_443nm', 'AOD_Extinction_Fine_667nm', 'AOD_Extinction_Fine_865nm', 'AOD_Extinction_Fine_1020nm', 'AOD_Extinction_Coarse_443nm', 'AOD_Extinction_Coarse_667nm', 'AOD_Extinction_Coarse_865nm', 'AOD_Extinction_Coarse_1020nm', 'Extinction_Angstrom_Exponent_440_870nm_Total', 'Single_Scattering_Albedo_443nm', 'Single_Scattering_Albedo_667nm', 'Single_Scattering_Albedo_865nm', 'Single_Scattering_Albedo_1020nm', 'Absorption_AOD_443nm', 'Absorption_AOD_667nm', 'Absorption_AOD_865nm', 'Absorption_AOD_1020nm', 'Absorption_Angstrom_Exponent_440_870nm', 'Refractive_Index_Real_Part_443nm', 'Refractive_Index_Real_Part_667nm', 'Refractive_Inde